In [ ]:
"""
================================================================================
LEVEL 1 - TASK 2: LINEAR REGRESSION MODEL
================================================================================
Objective: Predict house prices using number of rooms
Dataset: Boston Housing Dataset
================================================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Set display options
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print("="*80)
print("LEVEL 1 - TASK 2: LINEAR REGRESSION MODEL")
print("="*80)

# ============================================================================
# STEP 1: SETUP PATHS
# ============================================================================

print("\n" + "="*80)
print("STEP 1: SETUP PATHS")
print("="*80)

current_dir = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
main_dir = os.path.dirname(current_dir)
dataset_path = os.path.join(main_dir, 'datasets', '4) house Prediction Data Set.csv')
output_dir = os.path.join(main_dir, 'outputs')
image_dir = os.path.join(main_dir, 'images')

os.makedirs(output_dir, exist_ok=True)
os.makedirs(image_dir, exist_ok=True)

print(f"📁 Dataset path: {dataset_path}")
print(f"📁 Output directory: {output_dir}")

# ============================================================================
# STEP 2: LOAD DATASET
# ============================================================================

print("\n" + "="*80)
print("STEP 2: LOAD DATASET")
print("="*80)

# Column names for housing dataset
column_names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 
                'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT', 'MEDV']

if os.path.exists(dataset_path):
    housing_df = pd.read_csv(dataset_path, header=None, sep='\s+')
    housing_df.columns = column_names
    print(f"✅ Loaded from: {dataset_path}")
else:
    # Try alternative paths
    alt_paths = [
        '../datasets/4) house Prediction Data Set.csv',
        './datasets/4) house Prediction Data Set.csv',
        '4) house Prediction Data Set.csv'
    ]
    
    file_found = False
    for path in alt_paths:
        if os.path.exists(path):
            housing_df = pd.read_csv(path, header=None, sep='\s+')
            housing_df.columns = column_names
            print(f"✅ Loaded from: {path}")
            file_found = True
            break
    
    if not file_found:
        print("⚠️ Housing dataset not found. Creating sample data...")
        np.random.seed(42)
        n_samples = 506
        housing_df = pd.DataFrame({
            'RM': np.random.normal(6.0, 0.8, n_samples),
            'MEDV': np.random.normal(22.0, 8.0, n_samples)
        })
        housing_df['MEDV'] = housing_df['MEDV'] + (housing_df['RM'] - 6.0) * 5

print(f"\n📌 DATASET SHAPE: {housing_df.shape[0]} rows × {housing_df.shape[1]} columns")
print("\n📌 FIRST 5 ROWS:")
print(housing_df.head())

# ============================================================================
# STEP 3: SELECT FEATURES AND TARGET
# ============================================================================

print("\n" + "="*80)
print("STEP 3: SELECT FEATURES AND TARGET")
print("="*80)

X = housing_df[['RM']]  # Number of rooms
y = housing_df['MEDV']   # House price

print(f"📌 FEATURE (X): RM - Average number of rooms per dwelling")
print(f"   Range: {X['RM'].min():.2f} to {X['RM'].max():.2f} rooms")
print(f"\n📌 TARGET (y): MEDV - Median house value in $1000s")
print(f"   Range: ${y.min()*1000:.0f} to ${y.max()*1000:.0f}")
print(f"   Average: ${y.mean()*1000:.0f}")

# Visualize relationship
plt.figure(figsize=(10, 6))
plt.scatter(X, y, alpha=0.5, color='blue')
plt.xlabel('Average Number of Rooms (RM)')
plt.ylabel('Median House Value ($1000s)')
plt.title('Relationship: Number of Rooms vs House Price')
plt.grid(True, alpha=0.3)
plt.savefig(os.path.join(image_dir, 'housing_scatter.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Saved: {os.path.join(image_dir, 'housing_scatter.png')}")

# ============================================================================
# STEP 4: SPLIT DATA
# ============================================================================

print("\n" + "="*80)
print("STEP 4: SPLIT DATA")
print("="*80)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"📌 Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"📌 Testing set: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)")

# ============================================================================
# STEP 5: TRAIN LINEAR REGRESSION MODEL
# ============================================================================

print("\n" + "="*80)
print("STEP 5: TRAIN LINEAR REGRESSION MODEL")
print("="*80)

model = LinearRegression()
model.fit(X_train, y_train)

intercept = model.intercept_
coefficient = model.coef_[0]
price_per_room = coefficient * 1000

print(f"✅ Model trained successfully!")
print(f"\n📌 Model Equation: Price = {intercept:.2f} + {coefficient:.2f} × Rooms")
print(f"\n📌 INTERPRETATION:")
print(f"   • Base price (0 rooms): ${intercept*1000:.0f}")
print(f"   • Each additional room increases price by ${price_per_room:.2f}")

# ============================================================================
# STEP 6: MAKE PREDICTIONS
# ============================================================================

print("\n" + "="*80)
print("STEP 6: MAKE PREDICTIONS")
print("="*80)

y_pred = model.predict(X_test)

print("\n📌 SAMPLE PREDICTIONS (First 10 test samples):")
print("-" * 65)
print(f"{'Actual Price':>15} {'Predicted Price':>18} {'Difference':>15}")
print("-" * 65)

for i in range(10):
    actual = y_test.iloc[i] * 1000
    predicted = y_pred[i] * 1000
    diff = actual - predicted
    print(f"${actual:>13,.0f}  ${predicted:>15,.0f}  ${diff:>12,.0f}")

# ============================================================================
# STEP 7: EVALUATE MODEL
# ============================================================================

print("\n" + "="*80)
print("STEP 7: MODEL EVALUATION")
print("="*80)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

rmse_dollars = rmse * 1000
explained_variance = r2 * 100

print(f"\n📌 EVALUATION METRICS:")
print(f"   • Mean Squared Error (MSE): {mse:.2f}")
print(f"   • Root Mean Squared Error (RMSE): ${rmse_dollars:.2f}")
print(f"   • R-squared (R²) Score: {r2:.4f}")

print(f"\n📌 INTERPRETATION:")
print(f"   • R² = {r2:.4f} means the model explains {explained_variance:.1f}% of price variation")
print(f"   • Average prediction error: ${rmse_dollars:.2f}")

# ============================================================================
# STEP 8: VISUALIZE RESULTS
# ============================================================================

print("\n" + "="*80)
print("STEP 8: VISUALIZE RESULTS")
print("="*80)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Regression line
axes[0].scatter(X_test, y_test, alpha=0.5, color='blue', label='Actual')
axes[0].plot(X_test, y_pred, color='red', linewidth=2, label='Predicted')
axes[0].set_xlabel('Average Number of Rooms')
axes[0].set_ylabel('House Price ($1000s)')
axes[0].set_title('Linear Regression: Actual vs Predicted')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Residuals
residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.5, color='green')
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted Values')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(image_dir, 'linear_regression_results.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Saved: {os.path.join(image_dir, 'linear_regression_results.png')}")

# ============================================================================
# STEP 9: SAVE RESULTS
# ============================================================================

print("\n" + "="*80)
print("STEP 9: SAVE RESULTS")
print("="*80)

results_df = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_pred,
    'Residual': residuals
})
results_df.to_csv(os.path.join(output_dir, 'linear_regression_results.csv'), index=False)
print(f"✅ Saved: {os.path.join(output_dir, 'linear_regression_results.csv')}")

# ============================================================================
# STEP 10: SUMMARY
# ============================================================================

print("\n" + "="*80)
print("STEP 10: SUMMARY")
print("="*80)

print(f"""
╔════════════════════════════════════════════════════════════════════════════╗
║                    LEVEL 1 - TASK 2 COMPLETED                             ║
╠════════════════════════════════════════════════════════════════════════════╣
║                                                                            ║
║  ✅ Linear Regression Model for House Price Prediction                    ║
║                                                                            ║
║  MODEL EQUATION:                                                          ║
║  ──────────────────────────────────────────────────────────────────────── ║
║  Price = {intercept:.2f} + {coefficient:.2f} × Rooms                      ║
║                                                                            ║
║  PERFORMANCE:                                                             ║
║  ──────────────────────────────────────────────────────────────────────── ║
║  • R² Score: {r2:.4f}                                                     ║
║  • RMSE: ${rmse_dollars:.2f}                                              ║
║                                                                            ║
║  OUTPUT FILES:                                                            ║
║  ──────────────────────────────────────────────────────────────────────── ║
║  • outputs/linear_regression_results.csv                                  ║
║  • images/linear_regression_results.png                                   ║
║  • images/housing_scatter.png                                             ║
║                                                                            ║
╚════════════════════════════════════════════════════════════════════════════╝
""")

print("\n🎉 LINEAR REGRESSION COMPLETED SUCCESSFULLY!")